# This notebook builds the **image-retrieval prototype** for Case Study 1.
---

## Expected folder structure

```text
dinov2_prototype.ipynb

data/
  
  true_mapping.xlsx
  dataset_dev/
    gallery/
      object_001/
        img_001.jpg
        img_002.jpg
      object_002/
        img_001.jpg
    query/
      img_001.jpg
      img_002.jpg
      ...
```

In [ ]:
%pip install -q pandas openpyxl pillow numpy faiss-cpu torch torchvision transformers tqdm scikit-learn matplotlib

In [37]:
from pathlib import Path
import re
import json
from collections import defaultdict

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch
import faiss
import matplotlib.pyplot as plt

from transformers import AutoImageProcessor, AutoModel

# Fixed best parameters
BEST_SCORE_THRESHOLD = 0.50
BEST_MARGIN_THRESHOLD = 0.03

# Retrieval settings
TOP_K_IMAGES = 30
TOP_K_OBJECTS = 5
TOP_N_PER_OBJECT = 2

## 1. Configure paths

In [ ]:
# Change this to your local path before running.
PROJECT_ROOT = Path("./data").resolve()

OUTPUTS_DIR = Path("./dinov2_prototype_output").resolve()
OUTPUTS_DIR.mkdir(exist_ok=True)

EMBEDDINGS_PATH = OUTPUTS_DIR / "gallery_embeddings.npy"
INDEX_PATH = OUTPUTS_DIR / "gallery_faiss.index"
METADATA_PATH = OUTPUTS_DIR / "gallery_index_metadata.csv"
METRICS_PATH = OUTPUTS_DIR / "metrics_fixed.json"
RESULTS_PATH = OUTPUTS_DIR / "query_results_fixed.csv"

print("Outputs folder:", OUTPUTS_DIR)

DATASET_DIR = PROJECT_ROOT / "dataset_dev"
GALLERY_DIR = DATASET_DIR / "gallery"
QUERY_DIR = DATASET_DIR / "query"
MAPPING_XLSX = PROJECT_ROOT / "true_mapping.xlsx"

MODEL_NAME = "facebook/dinov2-base"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

use_cache = (
    EMBEDDINGS_PATH.exists() and
    INDEX_PATH.exists() and
    METADATA_PATH.exists()
)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("GALLERY_DIR exists:", GALLERY_DIR.exists(), GALLERY_DIR)
print("QUERY_DIR exists:", QUERY_DIR.exists(), QUERY_DIR)
print("MAPPING_XLSX exists:", MAPPING_XLSX.exists(), MAPPING_XLSX)
print("DEVICE:", DEVICE)

## 2. Helper functions

In [39]:
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}

def is_image_file(path: Path) -> bool:
    return path.suffix.lower() in IMAGE_EXTS

def numeric_suffix(text: str):
    """
    Returns the last integer found in a string, or None.
    Examples:
      object_066 -> 66
      query_img_1 -> 1
      img_001 -> 1
    """
    matches = re.findall(r"(\d+)", str(text))
    return int(matches[-1]) if matches else None

def normalize_object_id(value):
    """
    Converts values like:
      7 -> object_007
      '66' -> object_066
      'object_7' -> object_007
      'none' -> None
    """
    if pd.isna(value):
        return None

    s = str(value).strip()
    if s.lower() == "none":
        return None

    n = numeric_suffix(s)
    if n is None:
        return None

    return f"object_{n:03d}"

def l2_normalize(vectors: np.ndarray) -> np.ndarray:
    norms = np.linalg.norm(vectors, axis=1, keepdims=True)
    norms = np.maximum(norms, 1e-12)
    return vectors / norms

def cosine_from_l2_distance(l2_dist: float) -> float:
    """
    If vectors are L2-normalized, squared L2 distance and cosine similarity relate as:
      ||x-y||^2 = 2 - 2*cos(x,y)
      cos(x,y) = 1 - d^2/2
    """
    return 1.0 - (l2_dist / 2.0)

def show_image(path, title=None, figsize=(4, 4)):
    img = Image.open(path).convert("RGB")
    plt.figure(figsize=figsize)
    plt.imshow(img)
    plt.axis("off")
    if title:
        plt.title(title)
    plt.show()

## 3. Load gallery and query images

In [40]:
if not use_cache:
    gallery_records = []
    gallery_object_dirs = sorted([p for p in GALLERY_DIR.iterdir() if p.is_dir()])

    for object_dir in gallery_object_dirs:
        object_id = object_dir.name
        for img_path in sorted(object_dir.rglob("*")):
            if img_path.is_file() and is_image_file(img_path):
                gallery_records.append({
                    "object_id": object_id,
                    "image_path": str(img_path),
                    "image_name": img_path.name,
                })

    gallery_df = pd.DataFrame(gallery_records)

    query_records = []
    for img_path in sorted(QUERY_DIR.rglob("*")):
        if img_path.is_file() and is_image_file(img_path):
            query_records.append({
                "query_path": str(img_path),
                "query_name": img_path.name,
                "query_stem": img_path.stem,
                "query_num": numeric_suffix(img_path.stem),
            })

    query_df = pd.DataFrame(query_records)

    print(f"Gallery objects: {gallery_df['object_id'].nunique() if not gallery_df.empty else 0}")
    print(f"Gallery images:  {len(gallery_df)}")
    print(f"Query images:    {len(query_df)}")

    display(gallery_df.drop(columns=["image_path"], errors="ignore").head())
    display(query_df.drop(columns=["query_path"], errors="ignore").head())
else:
    print("Loading cached gallery metadata...")
    gallery_df = pd.read_csv(METADATA_PATH)
    print(f"Gallery objects: {gallery_df['object_id'].nunique() if not gallery_df.empty else 0}")
    print(f"Gallery images:  {len(gallery_df)}")
    display(gallery_df.drop(columns=["image_path"], errors="ignore").head())

    

Loading cached gallery metadata...
Gallery objects: 74
Gallery images:  1088


,object_id,image_name
0,object_001,img_001.jpg
1,object_001,img_002.jpg
2,object_001,img_003.jpg
3,object_001,img_004.jpg
4,object_001,img_005.jpg


## 4. Load and normalize `true_mapping.xlsx`

This code tries to infer the two important columns:
- query identifier
- target object identifier or `none`

In [41]:
mapping_raw = pd.read_excel(MAPPING_XLSX)
display(mapping_raw.head())

if mapping_raw.shape[1] < 2:
    raise ValueError("Expected at least 2 columns in true_mapping.xlsx")

# Take the first two columns by default
mapping = mapping_raw.iloc[:, :2].copy()
mapping.columns = ["mapping_query_key", "mapping_target_raw"]

mapping["mapping_query_key"] = mapping["mapping_query_key"].astype(str).str.strip()
mapping["target_object_id"] = mapping["mapping_target_raw"].apply(normalize_object_id)
mapping["mapping_query_num"] = mapping["mapping_query_key"].apply(numeric_suffix)

display(mapping.head(10))
print("Known targets:", mapping["target_object_id"].notna().sum())
print("Unknown/none targets:", mapping["target_object_id"].isna().sum())

,query_img_1,66
0,query_img_2,60
1,query_img_3,none
2,query_img_4,7
3,query_img_5,none
4,query_img_6,34


,mapping_query_key,mapping_target_raw,target_object_id,mapping_query_num
0,query_img_2,60,object_060,2
1,query_img_3,none,None,3
2,query_img_4,7,object_007,4
3,query_img_5,none,None,5
4,query_img_6,34,object_034,6
5,query_img_7,33,object_033,7
6,query_img_8,18,object_018,8
7,query_img_9,17,object_017,9
8,query_img_10,62,object_062,10
9,query_img_11,18,object_018,11


Known targets: 70
Unknown/none targets: 29


## 5. Match mapping rows to actual query image files

This resolver tries the following in this order:

1. exact filename match
2. exact stem match
3. numeric suffix match (for cases like `query_img_1` <-> `img_001.jpg`)

In [42]:
query_name_to_path = dict(zip(query_df["query_name"], query_df["query_path"]))
query_stem_to_path = dict(zip(query_df["query_stem"], query_df["query_path"]))

# numeric suffix -> list of query paths
query_num_to_paths = defaultdict(list)
for _, row in query_df.iterrows():
    if row["query_num"] is not None:
        query_num_to_paths[int(row["query_num"])].append(row["query_path"])

resolved_query_paths = []
resolution_method = []

for _, row in mapping.iterrows():
    key = row["mapping_query_key"]
    key_num = row["mapping_query_num"]

    path = None
    method = None

    # 1) exact filename match
    if key in query_name_to_path:
        path = query_name_to_path[key]
        method = "exact_filename"

    # 2) exact stem match
    elif key in query_stem_to_path:
        path = query_stem_to_path[key]
        method = "exact_stem"

    # 3) numeric suffix match
    elif key_num is not None and len(query_num_to_paths.get(int(key_num), [])) == 1:
        path = query_num_to_paths[int(key_num)][0]
        method = "numeric_suffix"

    resolved_query_paths.append(path)
    resolution_method.append(method)

mapping["query_path"] = resolved_query_paths
mapping["resolution_method"] = resolution_method

display(mapping.drop(columns=["query_path"], errors="ignore").head(20))

unresolved = mapping[mapping["query_path"].isna()].copy()
print("Resolved rows:", len(mapping) - len(unresolved))
print("Unresolved rows:", len(unresolved))

if len(unresolved):
    display(unresolved.drop(columns=["query_path"], errors="ignore").head(20))
    print("Some mapping rows could not be matched to query files.")
    print("If needed, manually create a mapping between Excel keys and query filenames.")

,mapping_query_key,mapping_target_raw,target_object_id,mapping_query_num,resolution_method
0,query_img_2,60,object_060,2,numeric_suffix
1,query_img_3,none,None,3,numeric_suffix
2,query_img_4,7,object_007,4,numeric_suffix
3,query_img_5,none,None,5,numeric_suffix
4,query_img_6,34,object_034,6,numeric_suffix
5,query_img_7,33,object_033,7,numeric_suffix
6,query_img_8,18,object_018,8,numeric_suffix
7,query_img_9,17,object_017,9,numeric_suffix
8,query_img_10,62,object_062,10,numeric_suffix
9,query_img_11,18,object_018,11,numeric_suffix


Resolved rows: 86
Unresolved rows: 13


,mapping_query_key,mapping_target_raw,target_object_id,mapping_query_num,resolution_method
70,query_img_72,36,object_036,72,None
87,query_img_89,NaN,None,89,None
88,query_img_90,NaN,None,90,None
89,query_img_91,NaN,None,91,None
90,query_img_92,NaN,None,92,None
91,query_img_93,NaN,None,93,None
92,query_img_94,NaN,None,94,None
93,query_img_95,NaN,None,95,None
94,query_img_96,NaN,None,96,None
95,query_img_97,NaN,None,97,None


Some mapping rows could not be matched to query files.
If needed, manually create a mapping between Excel keys and query filenames.


## 6. Build the evaluation table

In [43]:
eval_df = mapping.dropna(subset=["query_path"]).copy()
eval_df["query_name"] = eval_df["query_path"].apply(lambda p: Path(p).name)
eval_df["is_known"] = eval_df["target_object_id"].notna()

print("Evaluation rows:", len(eval_df))
print("Known queries:", int(eval_df["is_known"].sum()))
print("Unknown queries:", int((~eval_df["is_known"]).sum()))

display(eval_df.drop(columns=["query_path"], errors="ignore").head(10))

Evaluation rows: 86
Known queries: 69
Unknown queries: 17


,mapping_query_key,mapping_target_raw,target_object_id,mapping_query_num,resolution_method,query_name,is_known
0,query_img_2,60,object_060,2,numeric_suffix,img_002.JPG,True
1,query_img_3,none,None,3,numeric_suffix,img_003.JPG,False
2,query_img_4,7,object_007,4,numeric_suffix,img_004.JPG,True
3,query_img_5,none,None,5,numeric_suffix,img_005.JPG,False
4,query_img_6,34,object_034,6,numeric_suffix,img_006.JPG,True
5,query_img_7,33,object_033,7,numeric_suffix,img_007.JPG,True
6,query_img_8,18,object_018,8,numeric_suffix,img_008.JPG,True
7,query_img_9,17,object_017,9,numeric_suffix,img_009.JPG,True
8,query_img_10,62,object_062,10,numeric_suffix,img_010.JPG,True
9,query_img_11,18,object_018,11,numeric_suffix,img_011.JPG,True


## 7. Load DINOv2

Using the pretrained DINOv2 model as a **feature extractor**.

In [44]:
processor = AutoImageProcessor.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE)
model.eval()

print("Loaded model:", MODEL_NAME)

Loaded model: facebook/dinov2-base


## 8. Embedding functions

In [45]:
from torch.utils.data import Dataset, DataLoader

class ImagePathDataset(Dataset):
    def __init__(self, image_paths):
        self.image_paths = list(image_paths)

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        path = self.image_paths[idx]
        with Image.open(path) as img:
            img = img.convert("RGB")
        return img, path

def collate_images_and_paths(batch):
    images = [item[0] for item in batch]
    paths = [item[1] for item in batch]
    return images, paths

@torch.inference_mode()
def embed_image_paths_parallel(image_paths, batch_size=32, num_workers=4):
    dataset = ImagePathDataset(image_paths)

    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        collate_fn=collate_images_and_paths,
        persistent_workers=(num_workers > 0),
        prefetch_factor=2 if num_workers > 0 else None,
    )

    all_embeds = []

    for images, batch_paths in tqdm(loader, desc="Embedding gallery"):
        inputs = processor(images=images, return_tensors="pt")
        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

        outputs = model(**inputs)

        if hasattr(outputs, "pooler_output") and outputs.pooler_output is not None:
            batch_embeds = outputs.pooler_output
        else:
            batch_embeds = outputs.last_hidden_state.mean(dim=1)

        batch_embeds = batch_embeds.detach().to("cpu").float().numpy()
        all_embeds.append(batch_embeds)

    all_embeds = np.concatenate(all_embeds, axis=0).astype("float32")
    all_embeds = l2_normalize(all_embeds)
    return all_embeds

## 9. Embed gallery images and build a FAISS index

In [46]:
# Cache-aware gallery embedding + FAISS index builder/loader



use_cache = (
    EMBEDDINGS_PATH.exists() and
    INDEX_PATH.exists() and
    METADATA_PATH.exists()
)

if use_cache:
    print("Loading cached gallery embeddings and FAISS index...")
    gallery_embeddings = np.load(EMBEDDINGS_PATH).astype("float32")
    gallery_df = pd.read_csv(METADATA_PATH)
    index = faiss.read_index(str(INDEX_PATH))

    print("Loaded cache.")
    print("gallery_embeddings shape:", gallery_embeddings.shape)
    print("FAISS index ntotal:", index.ntotal)

else:
    gallery_paths = gallery_df["image_path"].tolist()
    print("No cache found. Building gallery embeddings and FAISS index...")

    gallery_embeddings = embed_image_paths_parallel(
        gallery_paths,
        batch_size=32 if DEVICE == "mps" else 16,
        num_workers=4 if DEVICE == "cuda" else 0
    )

    embedding_dim = gallery_embeddings.shape[1]
    index = faiss.IndexFlatL2(embedding_dim)
    index.add(gallery_embeddings)

    np.save(EMBEDDINGS_PATH, gallery_embeddings)
    gallery_df.to_csv(METADATA_PATH, index=False)
    faiss.write_index(index, str(INDEX_PATH))

    print("Built and saved cache.")
    print("gallery_embeddings shape:", gallery_embeddings.shape)
    print("FAISS index ntotal:", index.ntotal)

Loading cached gallery embeddings and FAISS index...
Loaded cache.
gallery_embeddings shape: (1088, 768)
FAISS index ntotal: 1088


## 10. Retrieval helpers

Retrieving at the **image level**, then aggregating to the **object level**.

Baseline aggregation rule:
- per object, keep the **best** similarity score among its matched images

In [47]:
from collections import defaultdict
import numpy as np

def retrieve_objects_for_query(query_path, top_k_images=30, top_k_objects=5, top_n_per_object=2):
    query_embed = embed_image_paths_parallel([query_path], batch_size=1, num_workers=0)
    distances, indices = index.search(query_embed, top_k_images)

    distances = distances[0]
    indices = indices[0]

    image_matches = []
    object_scores = defaultdict(list)

    for dist, idx in zip(distances, indices):
        if idx < 0:
            continue

        rec = gallery_df.iloc[int(idx)]
        object_id = rec["object_id"]
        score = cosine_from_l2_distance(float(dist))

        image_matches.append({
            "object_id": object_id,
            "gallery_path": rec["image_path"],
            "gallery_name": rec["image_name"],
            "score": score,
        })

        object_scores[object_id].append(score)

    ranked_objects = []
    for object_id, scores in object_scores.items():
        scores = sorted(scores, reverse=True)
        agg_score = float(np.mean(scores[:top_n_per_object]))

        ranked_objects.append({
            "object_id": object_id,
            "score": agg_score,
            "num_matches": len(scores),
            "best_score": scores[0],
        })

    ranked_objects = sorted(
        ranked_objects,
        key=lambda x: x["score"],
        reverse=True
    )[:top_k_objects]

    return ranked_objects, image_matches

def predict_object_or_none(
    query_path,
    score_threshold=0.66,
    margin_threshold=0.03,
    top_k_images=30,
    top_k_objects=5,
    top_n_per_object=2,
):
    ranked_objects, image_matches = retrieve_objects_for_query(
        query_path=query_path,
        top_k_images=top_k_images,
        top_k_objects=top_k_objects,
        top_n_per_object=top_n_per_object,
    )

    if not ranked_objects:
        return None, ranked_objects, image_matches

    best_object = ranked_objects[0]["object_id"]
    best_score = ranked_objects[0]["score"]
    second_score = ranked_objects[1]["score"] if len(ranked_objects) > 1 else -1.0
    margin = best_score - second_score

    if best_score < score_threshold:
        return None, ranked_objects, image_matches

    if margin < margin_threshold:
        return None, ranked_objects, image_matches

    return best_object, ranked_objects, image_matches

## 11. Try one query manually

In [56]:
sample_query_path = eval_df.iloc[0]["query_path"]
sample_true = eval_df.iloc[0]["target_object_id"]

#print("Sample query:", sample_query_path)
print("True target:", sample_true)

pred, ranked_objects, image_matches = predict_object_or_none(sample_query_path)

print("Predicted:", pred)
print("Top objects:")
for r in ranked_objects[:5]:
    print(r)

# show_image(sample_query_path, title=f"Query: {Path(sample_query_path).name}")
# for match in image_matches[:3]:
#     show_image(match["gallery_path"], title=f"{match['object_id']} | score={match['score']:.3f}")

True target: object_060


Embedding gallery: 100%|██████████| 1/1 [00:00<00:00,  5.59it/s]

Predicted: object_060
Top objects:
{'object_id': 'object_060', 'score': 0.7146497666835785, 'num_matches': 17, 'best_score': 0.7467067241668701}
{'object_id': 'object_003', 'score': 0.646845281124115, 'num_matches': 1, 'best_score': 0.646845281124115}
{'object_id': 'object_002', 'score': 0.6431731432676315, 'num_matches': 2, 'best_score': 0.677587628364563}
{'object_id': 'object_073', 'score': 0.6321077942848206, 'num_matches': 2, 'best_score': 0.6482726335525513}
{'object_id': 'object_015', 'score': 0.62471604347229, 'num_matches': 1, 'best_score': 0.62471604347229}


## 12. Evaluate across all queries for one threshold

In [49]:
def evaluate_thresholds(
    eval_df,
    score_threshold=0.66,
    margin_threshold=0.03,
    top_k_images=30,
    top_k_objects=5,
    top_n_per_object=2,
):
    rows = []

    for _, row in tqdm(eval_df.iterrows(), total=len(eval_df),
                       desc=f"Evaluating score={score_threshold:.2f}, margin={margin_threshold:.2f}"):
        query_path = row["query_path"]
        true_object = row["target_object_id"]
        is_known = row["is_known"]

        pred_object, ranked_objects, _ = predict_object_or_none(
            query_path=query_path,
            score_threshold=score_threshold,
            margin_threshold=margin_threshold,
            top_k_images=top_k_images,
            top_k_objects=top_k_objects,
            top_n_per_object=top_n_per_object,
        )

        ranked_ids = [r["object_id"] for r in ranked_objects]
        best_score = ranked_objects[0]["score"] if ranked_objects else None
        second_score = ranked_objects[1]["score"] if len(ranked_objects) > 1 else None
        margin = (best_score - second_score) if (best_score is not None and second_score is not None) else None

        top1_correct_known = bool(is_known and len(ranked_ids) >= 1 and ranked_ids[0] == true_object)
        top3_correct_known = bool(is_known and true_object in ranked_ids[:3])
        unknown_correct = bool((not is_known) and pred_object is None)
        overall_correct = bool((is_known and pred_object == true_object) or ((not is_known) and pred_object is None))

        rows.append({
            "query_path": query_path,
            "query_name": Path(query_path).name,
            "true_object": true_object,
            "is_known": is_known,
            "pred_object": pred_object,
            "best_score": best_score,
            "second_score": second_score,
            "margin": margin,
            "ranked_ids": ranked_ids,
            "top1_correct_known": top1_correct_known,
            "top3_correct_known": top3_correct_known,
            "unknown_correct": unknown_correct,
            "overall_correct": overall_correct,
        })

    results_df = pd.DataFrame(rows)

    known_mask = results_df["is_known"]
    unknown_mask = ~results_df["is_known"]

    metrics = {
        "score_threshold": score_threshold,
        "margin_threshold": margin_threshold,
        "n_total": int(len(results_df)),
        "n_known": int(known_mask.sum()),
        "n_unknown": int(unknown_mask.sum()),
        "top1_known": float(results_df.loc[known_mask, "top1_correct_known"].mean()) if known_mask.any() else np.nan,
        "top3_known": float(results_df.loc[known_mask, "top3_correct_known"].mean()) if known_mask.any() else np.nan,
        "unknown_rejection": float(results_df.loc[unknown_mask, "unknown_correct"].mean()) if unknown_mask.any() else np.nan,
        "overall_accuracy": float(results_df["overall_correct"].mean()) if len(results_df) else np.nan,
    }

    return metrics, results_df

# metrics, results_df = evaluate_thresholds(
#     eval_df,
#     score_threshold=0.66,
#     margin_threshold=0.03,
#     top_k_images=30,
#     top_k_objects=5,
#     top_n_per_object=2,
# )

# metrics

## 13. Search for a better unknown threshold (Disabled because the best parameters have been already found)

Because the given mapping contains `none`, a threshold to decide when the system should reject a query as unknown is needed. Also, we need to set a minimum confidence margin between top 1 and top 2 result.

This cell tries multiple Threshold & Margin values and reports the metrics.

In [50]:
#core_thresholds = np.arange(0.50, 0.81, 0.02)
#margin_thresholds = np.arange(0.00, 0.10, 0.01)

# all_metrics = []

# for score_thr in score_thresholds:
#     for margin_thr in margin_thresholds:
#         metrics, _ = evaluate_thresholds(
#             eval_df,
#             score_threshold=float(score_thr),
#             margin_threshold=float(margin_thr),
#             top_k_images=30,
#             top_k_objects=5,
#             top_n_per_object=2,
#         )
#         all_metrics.append(metrics)

# metrics_df = pd.DataFrame(all_metrics)
# display(metrics_df.sort_values(["overall_accuracy", "top1_known"], ascending=False).head(20))

# best_row = metrics_df.sort_values(
#     ["overall_accuracy", "top1_known", "unknown_rejection"],
#     ascending=False
# ).iloc[0]

# best_score_threshold = float(best_row["score_threshold"])
# best_margin_threshold = float(best_row["margin_threshold"])

# print("Best score threshold:", best_score_threshold)
# print("Best margin threshold:", best_margin_threshold)

# best_row

In [51]:
# plt.figure(figsize=(8, 5))
# plt.plot(metrics_df["threshold"], metrics_df["top1_known"], label="Top-1 known")
# plt.plot(metrics_df["threshold"], metrics_df["top3_known"], label="Top-3 known")
# plt.plot(metrics_df["threshold"], metrics_df["unknown_rejection"], label="Unknown rejection")
# plt.plot(metrics_df["threshold"], metrics_df["overall_accuracy"], label="Overall accuracy")
# plt.xlabel("Threshold")
# plt.ylabel("Score")
# plt.title("Threshold sweep")
# plt.legend()
# plt.grid(True)
# plt.show()

# pivot = metrics_df.pivot(
#     index="margin_threshold",
#     columns="score_threshold",
#     values="overall_accuracy"
# )

# plt.figure(figsize=(10, 6))
# plt.imshow(pivot.values, aspect="auto", origin="lower")
# plt.xticks(range(len(pivot.columns)), [f"{x:.2f}" for x in pivot.columns], rotation=90)
# plt.yticks(range(len(pivot.index)), [f"{y:.2f}" for y in pivot.index])
# plt.xlabel("score_threshold")
# plt.ylabel("margin_threshold")
# plt.title("Overall accuracy heatmap")
# plt.colorbar()
# plt.show()

## 14. Run final evaluation with the best threshold & best margin

In [52]:
final_metrics, final_results_df = evaluate_thresholds(
    eval_df,
    score_threshold=BEST_SCORE_THRESHOLD,
    margin_threshold=BEST_MARGIN_THRESHOLD,
    top_k_images=TOP_K_IMAGES,
    top_k_objects=TOP_K_OBJECTS,
    top_n_per_object=TOP_N_PER_OBJECT,
)

print(json.dumps(final_metrics, indent=2))
display(final_results_df.drop(columns=["query_path"], errors="ignore").head(20))

Evaluating score=0.50, margin=0.03: 100%|██████████| 86/86 [00:18<00:00,  4.77it/s]

{
  "score_threshold": 0.5,
  "margin_threshold": 0.03,
  "n_total": 86,
  "n_known": 69,
  "n_unknown": 17,
  "top1_known": 0.5362318840579711,
  "top3_known": 0.6086956521739131,
  "unknown_rejection": 0.8235294117647058,
  "overall_accuracy": 0.5
}


,query_name,true_object,is_known,pred_object,best_score,second_score,margin,ranked_ids,top1_correct_known,top3_correct_known,unknown_correct,overall_correct
0,img_002.JPG,object_060,True,object_060,0.714650,0.646845,0.067804,"[object_060, object_003, object_002, object_07...",True,True,False,True
1,img_003.JPG,None,False,None,0.557695,0.557071,0.000625,"[object_003, object_002, object_058, object_00...",False,False,True,True
2,img_004.JPG,object_007,True,None,0.708647,0.691454,0.017192,"[object_002, object_005, object_040, object_00...",False,False,False,False
3,img_005.JPG,None,False,None,0.736495,0.727312,0.009183,"[object_002, object_001, object_003, object_06...",False,False,True,True
4,img_006.JPG,object_034,True,None,0.555212,0.540378,0.014835,"[object_034, object_003, object_033, object_03...",True,True,False,False
5,img_007.JPG,object_033,True,object_033,0.585765,0.535290,0.050475,"[object_033, object_032, object_034, object_01...",True,True,False,True
6,img_008.JPG,object_018,True,None,0.718565,0.706918,0.011647,"[object_014, object_054, object_015, object_04...",False,False,False,False
7,img_009.JPG,object_017,True,object_017,0.814055,0.776093,0.037962,"[object_017, object_055, object_001, object_04...",True,True,False,True
8,img_010.JPG,object_062,True,None,0.843114,0.836191,0.006923,"[object_062, object_060, object_018, object_00...",True,True,False,False
9,img_011.JPG,object_018,True,object_018,0.868614,0.825282,0.043332,"[object_018, object_024, object_001, object_00...",True,True,False,True


## 15. Inspect errors

In [53]:
known_errors = final_results_df[
    (final_results_df["is_known"]) &
    (~final_results_df["top1_correct_known"])
].copy()

unknown_errors = final_results_df[
    (~final_results_df["is_known"]) &
    (~final_results_df["unknown_correct"])
].copy()

print("Known-query Top-1 errors:", len(known_errors))
print("Unknown-query rejection errors:", len(unknown_errors))

display(known_errors.drop(columns=["query_path"], errors="ignore").head(10))
display(unknown_errors.drop(columns=["query_path"], errors="ignore").head(10))

Known-query Top-1 errors: 32
Unknown-query rejection errors: 3


,query_name,true_object,is_known,pred_object,best_score,second_score,margin,ranked_ids,top1_correct_known,top3_correct_known,unknown_correct,overall_correct
2,img_004.JPG,object_007,True,None,0.708647,0.691454,0.017192,"[object_002, object_005, object_040, object_00...",False,False,False,False
6,img_008.JPG,object_018,True,None,0.718565,0.706918,0.011647,"[object_014, object_054, object_015, object_04...",False,False,False,False
10,img_012.JPG,object_044,True,None,0.739306,0.735493,0.003813,"[object_056, object_007, object_012, object_05...",False,False,False,False
11,img_013.JPG,object_043,True,None,0.813917,0.788675,0.025241,"[object_006, object_025, object_012, object_05...",False,False,False,False
12,img_014.JPG,object_046,True,object_054,0.812539,0.750974,0.061565,"[object_054, object_045, object_017, object_00...",False,False,False,False
13,img_015.JPG,object_049,True,object_054,0.787009,0.724532,0.062477,"[object_054, object_017, object_055, object_00...",False,False,False,False
14,img_016.JPG,object_048,True,None,0.683648,0.675536,0.008111,"[object_012, object_007, object_037, object_05...",False,False,False,False
15,img_017.JPG,object_050,True,None,0.697262,0.680453,0.016808,"[object_024, object_045, object_054, object_02...",False,False,False,False
16,img_018.JPG,object_020,True,None,0.680880,0.662726,0.018153,"[object_022, object_054, object_024, object_02...",False,False,False,False
17,img_019.JPG,object_051,True,object_055,0.765425,0.721382,0.044042,"[object_055, object_007, object_053, object_00...",False,False,False,False


,query_name,true_object,is_known,pred_object,best_score,second_score,margin,ranked_ids,top1_correct_known,top3_correct_known,unknown_correct,overall_correct
52,img_054.JPG,None,False,object_003,0.680974,0.600060,0.080914,"[object_003, object_034, object_005, object_00...",False,False,False,False
53,img_055.JPG,None,False,object_003,0.707936,0.614532,0.093403,"[object_003, object_002, object_012, object_00...",False,False,False,False
62,img_064.JPG,None,False,object_007,0.821526,0.780245,0.041281,"[object_007, object_060, object_018, object_00...",False,False,False,False


In [54]:
def inspect_result_row(
    result_row,
    score_threshold=0.66,
    margin_threshold=0.03,
    top_k_images=30,
    top_k_objects=5,
    top_n_per_object=2,
    n_gallery=5,
):
    query_path = result_row["query_path"]
    true_object = result_row["true_object"]

    pred_object, ranked_objects, image_matches = predict_object_or_none(
        query_path=query_path,
        score_threshold=score_threshold,
        margin_threshold=margin_threshold,
        top_k_images=top_k_images,
        top_k_objects=top_k_objects,
        top_n_per_object=top_n_per_object,
    )

    print("Query:", query_path)
    print("True:", true_object)
    print("Pred:", pred_object)
    print("Top objects:")
    for r in ranked_objects:
        print(
            f"  {r['object_id']}: agg={r['score']:.4f}, "
            f"best={r['best_score']:.4f}, matches={r['num_matches']}"
        )

    show_image(query_path, title=f"Query | true={true_object} | pred={pred_object}")

    shown = 0
    seen_paths = set()
    for match in image_matches:
        if match["gallery_path"] in seen_paths:
            continue
        seen_paths.add(match["gallery_path"])
        show_image(match["gallery_path"], title=f"{match['object_id']} | score={match['score']:.4f}")
        shown += 1
        if shown >= n_gallery:
            break

# Example:
# inspect_result_row(known_errors.iloc[0], score_threshold=best_score_threshold, margin_threshold=best_margin_threshold)

## 16. Save outputs

This saves:
- metrics
- per-query predictions

In [55]:
with open(METRICS_PATH, "w", encoding="utf-8") as f:
    json.dump(final_metrics, f, indent=2)

final_results_df.to_csv(RESULTS_PATH, index=False)

#print("Saved:", METRICS_PATH)
#print("Saved:", RESULTS_PATH)

## Future improvements

The next improvements are:

1. image preprocessing
   - center crop variants
   - background removal if needed

2. fine-tuning
   - use query <-> correct object mapping as supervised signal